# 🎙️ ECE22073 — Colab Pipeline Runtime

**Repo:** [victoras136/asr-notebook](https://github.com/victoras136/asr-notebook)

### Βήματα εκκίνησης
1. **Runtime → Change runtime type → T4 GPU**
2. Πρόσθεσε τα Secrets (🔑 εικονίδιο αριστερά): `OPENAI_API_KEY`, `HF_TOKEN`
3. Άλλαξε τις ρυθμίσεις στο **Cell 1** αν χρειαστεί
4. **Runtime → Run all** (`Ctrl+F9`)

> Τα cells τρέχουν με σειρά. Μην παραλείπεις κανένα.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION  (edit here, then Run all)         ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Podcast / TTS generation ─────────────────────────────────────
# Θέσε True μόνο αν χρειάζεσαι το Podcast page στο Streamlit.
# Προσθέτει 5-15 λεπτά install time + χρειάζεται GPU VRAM.
INSTALL_PODCAST_DEPS = False

# ── NeMo models (Nvidia Canary 1B, Nemotron) ─────────────────────
# Θέσε True για να ενεργοποιήσεις τα μοντέλα Canary / Nemotron.
# Προσθέτει ~5 λεπτά install time.
INSTALL_NEMO = False

# Ποιο TTS μοντέλο να κατεβάσει (αγνοείται αν INSTALL_PODCAST_DEPS=False):
#   "kokoro"  — γρήγορο, υψηλή ποιότητα, Apache 2.0  ~500 MB  ← προτεινόμενο
#   "dia"     — Dia-1.6B (Nari Labs), πολύ φυσικό διάλογος  ~3 GB
#   "bark"    — Suno Bark, εκφραστικό αλλά αργό             ~5 GB
#   "xtts"    — Coqui XTTS-v2, voice cloning                ~2 GB  (ακαδημαϊκή άδεια)
#   "f5"      — F5-TTS, τελευταία τεχνολογία                ~2 GB
PODCAST_TTS_MODEL = "kokoro"

# ─────────────────────────────────────────────────────────────────
print(f"Config: podcast_deps={INSTALL_PODCAST_DEPS}, nemo={INSTALL_NEMO}, tts='{PODCAST_TTS_MODEL}'")

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, subprocess
from pathlib import Path
from google.colab import userdata

# Secrets από το Colab Secrets panel (🔑 αριστερά)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"]       = userdata.get("HF_TOKEN")

# Clone ή pull του repo
os.chdir("/content")
REPO_DIR = Path("/content/asr-notebook")

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/victoras136/asr-notebook"
except Exception:
    repo_url = "https://github.com/victoras136/asr-notebook"

if not REPO_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", repo_url, str(REPO_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", repo_url], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

!find /content/asr-notebook -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null
sys.path.insert(0, str(REPO_DIR / 'Pipeline'))
print(f"✓ Repo ready at {REPO_DIR}")

In [ ]:
import warnings; warnings.filterwarnings('ignore')

print('📦 System deps...')
!apt-get install -y -q ffmpeg espeak-ng 2>&1 | tail -1

# uv: Rust-based pip replacement — parallel resolver, 10-100x faster than pip
print('📦 Installing uv...')
!pip install -q uv 2>&1 | tail -1

# Colab already has: torch, torchaudio, numpy, scipy, transformers, huggingface_hub
# uv resolves deps in parallel and skips already-satisfied packages instantly.
print('📦 Pipeline packages via uv (~2-3 min instead of 10+)...')
!uv pip install --system -q "numpy>=2.0.0" "numba>=0.60.0" "faster-whisper>=1.0.0" "pyannote.audio>=3.1.1" "pydub>=0.25.1" "openai>=1.30.0" "rouge-score>=0.1.2" "accelerate>=1.0.0" 2>&1 | tail -5

print('✓ Core deps ready')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import subprocess, sys

_DIA_COMMIT = "2811af1c5f47"  # last commit before June-27-2025 Transformers refactor

_TTS_CMDS = {
    "kokoro": ["pip", "install", "-q", "kokoro>=0.9.4"],
    "dia":    ["pip", "install", "-q", "--force-reinstall", f"git+https://github.com/nari-labs/dia.git@{_DIA_COMMIT}"],
    "bark":   ["pip", "install", "-q", "git+https://github.com/suno-ai/bark.git"],
    "xtts":   ["pip", "install", "-q", "git+https://github.com/coqui-ai/TTS.git"],
    "f5":     ["pip", "install", "-q", "git+https://github.com/SWivid/F5-TTS.git"],
}

if INSTALL_PODCAST_DEPS:
    if PODCAST_TTS_MODEL not in _TTS_CMDS:
        raise ValueError(f"Unknown TTS model '{PODCAST_TTS_MODEL}'. Choose: {list(_TTS_CMDS)}")
    print(f"📦 Installing TTS: {PODCAST_TTS_MODEL} (μπορεί να πάρει 5-15 λεπτά)...")
    r = subprocess.run(_TTS_CMDS[PODCAST_TTS_MODEL], capture_output=True, text=True)
    if r.stdout: print(r.stdout[-500:])
    if r.returncode != 0: print('STDERR:', r.stderr[-500:])
    print(f"✓ TTS ({PODCAST_TTS_MODEL}) installed")
else:
    print('⏭  Skipping podcast/TTS deps — αλλαξε INSTALL_PODCAST_DEPS=True στο Cell 1 αν θες')

# Always ensure dia is at the pre-refactor version if it's installed.
# The June-27-2025 refactor broke compatibility with nari-labs/Dia-1.6B Hub config.
# Also restore anyio>=4.0.0 because dia's gradio dep can downgrade it, breaking openai.
try:
    from dia.config import DiaConfig as _dc
    if 'decoder_config' in getattr(_dc, 'model_fields', {}):
        print(f"🔧 Incompatible dia version detected — pinning to {_DIA_COMMIT}...")
        r = subprocess.run(
            ["pip", "install", "-q", "--force-reinstall",
             f"git+https://github.com/nari-labs/dia.git@{_DIA_COMMIT}"],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print("❌ dia pin failed:", r.stderr[-200:])
        else:
            subprocess.run(["pip", "install", "-q", "anyio>=4.0.0"], capture_output=True)
            for _mod in list(sys.modules.keys()):
                if _mod.startswith('dia') or _mod in ('anyio', 'httpx'):
                    del sys.modules[_mod]
            print(f"✓ dia pinned to {_DIA_COMMIT}")
except ImportError:
    pass  # dia not installed yet — will be installed when a podcast job runs

# Always pin anyio>=4.0.0 regardless (guards against any other dep downgrading it)
subprocess.run(["pip", "install", "-q", "anyio>=4.0.0"], capture_output=True)

if INSTALL_NEMO:
    # Check if already installed to skip re-install
    try:
        import nemo
        print(f'✓ NeMo already installed ({nemo.__version__})')
    except ImportError:
        print('📦 Installing NeMo toolkit (Canary + Nemotron) — ~5 λεπτά...')
        r = subprocess.run(
            ['pip', 'install', 'nemo_toolkit[asr]'],
            capture_output=True, text=True
        )
        # Show last lines regardless of success — helps debug
        combined = (r.stdout + r.stderr).strip()
        if combined:
            print(combined[-1000:])
        if r.returncode != 0:
            print(f'❌ NeMo install failed (exit {r.returncode}) — Canary/Nemotron will be skipped')
        else:
            # Verify the import actually works
            try:
                import importlib
                if 'nemo' in sys.modules:
                    del sys.modules['nemo']
                importlib.import_module('nemo.collections.asr.models')
                print('✓ NeMo ready')
            except ImportError as e:
                print(f'⚠️  NeMo installed but import failed: {e}')
                print('   Try: Runtime → Restart and run all')
else:
    print('⏭  Skipping NeMo — αλλαξε INSTALL_NEMO=True στο Cell 1 για Canary/Nemotron')

In [ ]:
# Απαιτείται για pyannote.audio (speaker diarization — gated models)
import os
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('✓ HuggingFace authenticated via HF_TOKEN secret')
else:
    print('HF_TOKEN δεν βρέθηκε στα Secrets — συνδέσου interactively:')
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s", force=True)
import warnings; warnings.filterwarnings('ignore')
import os, sys
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/asr-notebook/Pipeline')
if '/content/asr-notebook/Pipeline' not in sys.path:
    sys.path.insert(0, '/content/asr-notebook/Pipeline')

import config
import drive_bridge as db
import colab_job_watcher as cjw

# main_loop: polls Drive, auto-exits after 5 min idle to save Colab runtime
cjw.main_loop()


## 📊 Monitoring

| | |
|---|---|
| **Drive output** | `MyDrive/ece22073/output/{job_id}/status.json` |
| **Streamlit polling** | κάθε 5 δευτερόλεπτα |
| **Stall detection** | αν δεν υπάρχει update σε 10 λεπτά → warning στο UI |
| **Μετά από disconnect** | ξανατρέξε cells 2–6 — τα input files μένουν στο Drive |

Για να σταματήσεις: **Runtime → Interrupt execution**

---
## 🔬 Evaluation (προαιρετικό)

Τρέξε το παρακάτω cell μόνο αν θέλεις να αξιολογήσεις WER/ROUGE.
Απαιτεί να έχει ήδη επεξεργαστεί ένα αρχείο από το pipeline.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json, sys
from pathlib import Path

sys.path.insert(0, '/content/asr-notebook/Benchmarks')

try:
    import jiwer, rouge_score
except ImportError:
    print('Missing: pip install jiwer rouge-score')
else:
    from evaluate_real_pipeline import run_real_evaluation
    success = run_real_evaluation()
    print(f"\nEvaluation {'PASSED' if success else 'FAILED'}")

    norm_path = Path('/content/asr-notebook/Results/normalized_transcript.txt')
    gt_path   = Path('/content/asr-notebook/Results/ground_truth.json')
    if norm_path.exists():
        norm_text = norm_path.read_text(encoding='utf-8')
        gt_text = ''
        if gt_path.exists():
            import json
            with open(gt_path) as f:
                gt_text = json.load(f).get("transcript", "")
        print("\n=== Ground Truth (first 300 chars) ===")
        print(gt_text[:300] or '(no GT file)')
        print("\n=== Normalized Output (first 300 chars) ===")
        print(norm_text[:300])
    else:
        print('normalized_transcript.txt not found — τρέξε πρώτα το pipeline')